# Recommendations with IBM

This notebook builds several recommender systems for IBM Watson Studio community articles: rank-based recommendations, user-user collaborative filtering, content-based recommendations with TF-IDF/KMeans, and SVD-based article similarity.


## Part I: Exploratory Data Analysis


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/user-item-interactions.csv").drop(columns=["Unnamed: 0"])
df_content = pd.read_csv("data/articles_community.csv").drop(columns=["Unnamed: 0"])
df_content = df_content.drop_duplicates(subset=["article_id"]).copy()
df["article_id"] = df["article_id"].astype(str)
df_content["article_id"] = df_content["article_id"].astype(str)

def email_mapper(frame):
    """Map anonymized email addresses to sequential integer user IDs."""
    coded_dict = {email: idx + 1 for idx, email in enumerate(frame["email"].drop_duplicates())}
    return frame["email"].map(coded_dict)

df["user_id"] = email_mapper(df)
del df["email"]
print(df.shape)
print(df_content.shape)


(45993, 3)
(1051, 6)


In [ ]:
interactions_by_user = df.groupby("user_id").size()
article_counts = df["article_id"].value_counts()
sol_1_dict = {
    "`50% of individuals have _____ or fewer interactions.`": int(interactions_by_user.median()),
    "`The total number of user-article interactions in the dataset is ______.`": int(df.shape[0]),
    "`The maximum number of user-article interactions by any 1 user is ______.`": int(interactions_by_user.max()),
    "`The most viewed article in the dataset was viewed _____ times.`": int(article_counts.max()),
    "`The article_id of the most viewed article is ______.`": str(article_counts.idxmax()),
    "`The number of unique articles that have at least 1 rating ______.`": int(df["article_id"].nunique()),
    "`The number of unique users in the dataset is ______`": int(df["user_id"].nunique()),
    "`The number of unique articles on the IBM platform`": int(df_content["article_id"].nunique()),
}
sol_1_dict


{
  "`50% of individuals have _____ or fewer interactions.`": 3,
  "`The total number of user-article interactions in the dataset is ______.`": 45993,
  "`The maximum number of user-article interactions by any 1 user is ______.`": 364,
  "`The most viewed article in the dataset was viewed _____ times.`": 937,
  "`The article_id of the most viewed article is ______.`": "1429.0",
  "`The number of unique articles that have at least 1 rating ______.`": 714,
  "`The number of unique users in the dataset is ______`": 5148,
  "`The number of unique articles on the IBM platform`": 1051
}


## Part II: Rank-Based Recommendations

Articles are ranked by interaction count because this dataset does not include explicit ratings.


In [ ]:
def get_top_article_ids(n, frame=df):
    """Return the IDs of the top n articles by interaction count."""
    return frame["article_id"].value_counts().head(n).index.tolist()

def get_top_articles(n, frame=df):
    """Return the titles of the top n articles by interaction count."""
    top_ids = get_top_article_ids(n, frame)
    titles = frame.drop_duplicates("article_id").set_index("article_id")["title"]
    return [titles.loc[article_id] for article_id in top_ids]

print(get_top_article_ids(5))
print(get_top_articles(5))


['1429.0', '1330.0', '1431.0', '1427.0', '1364.0']
['use deep learning for image classification', 'insights from new york car accident reports', 'visualize car data with brunel', 'use xgboost, scikit-learn & ibm watson machine learning apis', 'predicting churn with the spss random tree algorithm']


## Part III: User-User Collaborative Filtering


In [ ]:
def create_user_item_matrix(frame):
    """Create a binary matrix with users as rows and articles as columns."""
    user_item = frame.groupby(["user_id", "article_id"])["title"].count().unstack()
    return user_item.notna().astype(int)

user_item = create_user_item_matrix(df)
print(user_item.shape)


(5149, 714)


In [ ]:
def find_similar_users(user_id, matrix=user_item):
    """Find users ordered by descending dot-product similarity to user_id."""
    similarity = matrix.dot(matrix.loc[user_id]).drop(index=user_id)
    return similarity.sort_values(ascending=False, kind="mergesort").index.tolist()

def get_article_names(article_ids, frame=df):
    """Return article names associated with article IDs."""
    title_lookup = frame.drop_duplicates("article_id").set_index("article_id")["title"]
    return [title_lookup.loc[str(article_id)] for article_id in article_ids if str(article_id) in title_lookup.index]

def get_user_articles(user_id, matrix=user_item, frame=df):
    """Return IDs and names for articles already seen by a user."""
    article_ids = matrix.columns[matrix.loc[user_id] == 1].tolist()
    return article_ids, get_article_names(article_ids, frame)

similarity_check = {
    "The user that is most similar to user 1.": find_similar_users(1)[0],
    "The user that is the 10th most similar to user 131": find_similar_users(131)[9],
}
print(similarity_check)


{'The user that is most similar to user 1.': 3933, 'The user that is the 10th most similar to user 131': 242}


In [ ]:
def user_user_recs(user_id, m=10, matrix=user_item):
    """Recommend articles from similar users that the target user has not read."""
    seen_ids, _ = get_user_articles(user_id, matrix)
    recommendations = []
    for neighbor in find_similar_users(user_id, matrix):
        neighbor_ids, _ = get_user_articles(neighbor, matrix)
        for article_id in neighbor_ids:
            if article_id not in seen_ids and article_id not in recommendations:
                recommendations.append(article_id)
            if len(recommendations) >= m:
                return recommendations
    return recommendations

article_rank = df["article_id"].value_counts()
user_activity = user_item.sum(axis=1).sort_values(ascending=False)

def get_top_sorted_users(user_id, matrix=user_item):
    """Return similar users with similarity and activity tie-breakers."""
    similarity = matrix.dot(matrix.loc[user_id]).drop(index=user_id)
    neighbors = pd.DataFrame({
        "neighbor_id": similarity.index,
        "similarity": similarity.values,
        "num_interactions": user_activity.loc[similarity.index].values,
    })
    return neighbors.sort_values(by=["similarity", "num_interactions", "neighbor_id"], ascending=[False, False, True])

def user_user_recs_part2(user_id, m=10, matrix=user_item):
    """Improved CF recommendations ranked by neighbor similarity and article popularity."""
    seen_ids, _ = get_user_articles(user_id, matrix)
    recs = []
    for neighbor in get_top_sorted_users(user_id, matrix)["neighbor_id"]:
        neighbor_ids, _ = get_user_articles(neighbor, matrix)
        unseen = [article_id for article_id in neighbor_ids if article_id not in seen_ids]
        unseen = sorted(unseen, key=lambda article_id: article_rank.get(article_id, 0), reverse=True)
        for article_id in unseen:
            if article_id not in recs:
                recs.append(article_id)
            if len(recs) >= m:
                return recs, get_article_names(recs)
    return recs, get_article_names(recs)

print(user_user_recs_part2(1, 5))
print("New user fallback:", get_top_article_ids(5))


(['1330.0', '1364.0', '1314.0', '1162.0', '1304.0'], ['insights from new york car accident reports', 'predicting churn with the spss random tree algorithm', 'healthcare python streaming application demo', 'analyze energy consumption in buildings', 'gosales transactions for logistic regression model'])
New user fallback: ['1429.0', '1330.0', '1431.0', '1427.0', '1364.0']


## Part IV: Content-Based Recommendations

The content recommender uses article titles, descriptions, and bodies. TF-IDF creates text vectors and KMeans groups similar article content.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def make_content_model(content_frame=df_content, n_clusters=25):
    """Fit TF-IDF and KMeans article-content representations."""
    text = (
        content_frame["doc_full_name"].fillna("") + " " +
        content_frame["doc_description"].fillna("") + " " +
        content_frame["doc_body"].fillna("")
    )
    vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
    tfidf = vectorizer.fit_transform(text)
    clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = clusterer.fit_predict(tfidf)
    return vectorizer, tfidf, clusters

vectorizer, tfidf_matrix, content_clusters = make_content_model()
df_content["cluster"] = content_clusters
print("TF-IDF shape:", tfidf_matrix.shape)
print("Clusters:", df_content["cluster"].nunique())


TF-IDF shape: (1051, 3000)
Clusters: 25


In [ ]:
def make_content_recs(article_id, m=10, content_frame=df_content, tfidf=tfidf_matrix):
    """Recommend articles with the most similar text content to an article."""
    article_id = str(int(float(article_id))) if str(article_id).replace(".", "", 1).isdigit() else str(article_id)
    if article_id not in set(content_frame["article_id"]):
        return get_top_article_ids(m)
    idx = content_frame.index[content_frame["article_id"] == article_id][0]
    sims = cosine_similarity(tfidf[idx], tfidf).ravel()
    order = np.argsort(-sims)
    recs = []
    for pos in order:
        candidate = content_frame.iloc[pos]["article_id"]
        if candidate != article_id and candidate not in recs:
            recs.append(candidate)
        if len(recs) >= m:
            break
    return recs

print(make_content_recs("0", 5))


['182', '335', '355', '428', '416']


## Part V: Matrix Factorization

Because duplicate user-article interactions are converted to a binary matrix with no missing values, standard SVD can be applied directly. More latent features increase reconstruction accuracy, but 50 features is a practical balance for article-similarity recommendations.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

u, s, vt = np.linalg.svd(user_item.values)
article_ids = user_item.columns.tolist()
latent_features = np.arange(10, 701, 50)
reconstruction_acc = []
matrix = user_item.values
for k in latent_features:
    s_new = np.zeros((k, k))
    s_new[:k, :k] = np.diag(s[:k])
    preds = np.around(u[:, :k].dot(s_new).dot(vt[:k, :]))
    reconstruction_acc.append((preds == matrix).mean())

plt.figure(figsize=(7, 4))
plt.plot(latent_features, reconstruction_acc, marker="o")
plt.xlabel("Latent Features")
plt.ylabel("Reconstruction Accuracy")
plt.title("SVD Reconstruction Accuracy by Latent Features")
plt.tight_layout()
plt.savefig("svd_accuracy.png", dpi=150)
print("U, S, Vt shapes:", u.shape, s.shape, vt.shape)
print("Accuracy at first three latent settings:", reconstruction_acc[:3])


U, S, Vt shapes: (5149, 5149) (714,) (714, 714)
Accuracy at first three latent settings: [np.float64(0.9916796005642498), np.float64(0.9950500301110928), np.float64(0.9968836242984278)]


In [ ]:
def svd_article_recs(article_id, vt, article_ids, num_latent_features=50, m=10):
    """Recommend articles using cosine similarity in SVD article latent space."""
    article_id = str(article_id)
    if article_id not in article_ids:
        return get_top_article_ids(m)
    article_idx = article_ids.index(article_id)
    latent = vt[:num_latent_features, :].T
    sims = cosine_similarity(latent[article_idx].reshape(1, -1), latent).ravel()
    order = np.argsort(-sims)
    return [article_ids[i] for i in order if article_ids[i] != article_id][:m]

print(svd_article_recs("1429.0", vt, article_ids, 50, 5))


['1122.0', '1419.0', '1221.0', '60.0', '870.0']


## Results Discussion

Rank-based recommendations are strong for new users because they rely only on global interaction popularity. User-user collaborative filtering adds personalization when the user has prior interactions, but it suffers from the cold-start problem for new users. Content-based TF-IDF recommendations solve part of that issue at the article level by recommending semantically similar content. SVD gives a compact latent representation of articles and users; it is useful for article-article similarity, though online evaluation such as A/B testing click-through rate, dwell time, or repeat engagement would be needed before production deployment.
